# Phase 1 — ABCD EDA + BERTopic feasibility + fixtures

Review gate for the demo subset. This notebook **shows the failures**, not only the winner.

Offline labels (`flow` / `subflow`) are used here to *choose* the experiment and to *sanity-check* BERTopic. They are **not** runtime inputs. Runtime batches loaded at the end have no subflow fields.

In [1]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from collections import Counter, defaultdict
from pathlib import Path

import duckdb
import pyarrow as pa
from dotenv import load_dotenv
from IPython.display import display, JSON

def _find_root() -> Path:
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "src" / "playbook" / "config.py").exists():
            return candidate
    raise RuntimeError("Could not find worktree root containing src/playbook")


ROOT = _find_root()
os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from playbook.config import ABCD_JSON, GUIDELINES_JSON, KB_JSON, ONTOLOGY_JSON
from playbook.data import conversation_document, load_fixture_labels, load_week, parse_raw_conversation
from playbook.fixtures import DEMO_FLOW, ROLE_TO_SUBFLOW, WEEK_MEMBERSHIP
from playbook.topics import EXTRA_STOP, BertopicConfig, discover_topics

load_dotenv(ROOT / ".env")
con = duckdb.connect(database=":memory:")
print("ROOT", ROOT)
print("ABCD exists", ABCD_JSON.exists(), ABCD_JSON.stat().st_size if ABCD_JSON.exists() else 0)

ROOT C:\Users\doste\.cursor\worktrees\phase-01-data-topic-eda-a7c3e91b
ABCD exists True 121548569


## 1. Load ABCD into DuckDB

Conversations stay as JSON on disk; DuckDB is the query path for counts. Nested dialogue is parsed in Python only when we need excerpts or BERTopic documents.

In [2]:
raw = json.loads(ABCD_JSON.read_text(encoding="utf-8"))
index_rows = []
for split, convos in raw.items():
    for item in convos:
        index_rows.append(
            {
                "split": split,
                "conversation_id": str(item["convo_id"]),
                "flow": item["scenario"]["flow"],
                "subflow": item["scenario"]["subflow"],
            }
        )
con.register("abcd_index", pa.Table.from_pylist(index_rows))
display(con.execute("SELECT split, count(*) AS n FROM abcd_index GROUP BY 1 ORDER BY 1").df())
print("total", con.execute("SELECT count(*) FROM abcd_index").fetchone()[0])

ontology = json.loads(ONTOLOGY_JSON.read_text(encoding="utf-8"))
guidelines = json.loads(GUIDELINES_JSON.read_text(encoding="utf-8"))
kb = json.loads(KB_JSON.read_text(encoding="utf-8"))
print("ontology flows", ontology["intents"]["flows"])
print("n subflows listed", sum(len(v) for v in ontology["intents"]["subflows"].values()))

,split,n
0,dev,1004
1,test,1004
2,train,8034


total 10042
ontology flows ['account_access', 'manage_account', 'order_issue', 'product_defect', 'purchase_dispute', 'shipping_issue', 'single_item_query', 'storewide_query', 'subscription_inquiry', 'troubleshoot_site']
n subflows listed 55


In [3]:
display(JSON(guidelines, expanded=False))

<IPython.core.display.JSON object>

## 2. Flow landscape

Plan shortlist: `account_access` (3 subflows), `shipping_issue` (exactly 4), `product_defect` (6; only if we pick four). Avoid FAQ-like `storewide_query` / `single_item_query` unless nothing else works.

In [4]:
display(
    con.execute(
        """
        SELECT flow, count(*) AS n_convos, count(DISTINCT subflow) AS n_subflows
        FROM abcd_index
        GROUP BY 1
        ORDER BY n_convos DESC
        """
    ).df()
)

,flow,n_convos,n_subflows
0,storewide_query,1094,16
1,purchase_dispute,1076,8
2,product_defect,1070,6
3,account_access,1048,3
4,single_item_query,1045,32
5,order_issue,1040,9
6,troubleshoot_site,1026,4
7,shipping_issue,1020,4
8,subscription_inquiry,910,6
9,manage_account,713,8


## 3. Shortlist subflow counts + canonical actions

`kb.json` is the compact action-button list per subflow. Distinct sequences matter more than raw counts.

In [5]:
SHORTLIST = ["account_access", "shipping_issue", "product_defect"]
display(
    con.execute(
        """
        SELECT flow, subflow, count(*) AS n
        FROM abcd_index
        WHERE flow IN ('account_access', 'shipping_issue', 'product_defect')
        GROUP BY 1, 2
        ORDER BY flow, n DESC
        """
    ).df()
)
for flow in SHORTLIST:
    print(f"\n== {flow} kb actions ==")
    for subflow in ontology["intents"]["subflows"][flow]:
        print(f"  {subflow:20s} {kb[subflow]}")

,flow,subflow,n
0,account_access,recover_username,361
1,account_access,reset_2fa,351
2,account_access,recover_password,336
3,product_defect,return_size,191
4,product_defect,return_color,180
5,product_defect,refund_status,179
6,product_defect,refund_update,177
7,product_defect,refund_initiate,176
8,product_defect,return_stain,167
9,shipping_issue,status,268



== account_access kb actions ==
  recover_username     ['pull-up-account', 'verify-identity']
  recover_password     ['pull-up-account', 'enter-details', 'make-password']
  reset_2fa            ['pull-up-account', 'enter-details', 'send-link']

== shipping_issue kb actions ==
  status               ['pull-up-account', 'verify-identity', 'validate-purchase', 'ask-the-oracle', 'update-order']
  manage               ['pull-up-account', 'shipping-status', 'validate-purchase', 'update-order']
  missing              ['pull-up-account', 'validate-purchase', 'record-reason', 'update-order', 'make-purchase']
  cost                 ['pull-up-account', 'validate-purchase', 'shipping-status', 'update-order', 'offer-refund']

== product_defect kb actions ==
  refund_initiate      ['pull-up-account', 'validate-purchase', 'record-reason', 'enter-details', 'offer-refund']
  refund_update        ['pull-up-account', 'validate-purchase', 'record-reason', 'offer-refund']
  refund_status        ['pull-up-

Observed action sequences are noisier than the canonical KB lists (agents skip steps). Still, modes match the playbook.

In [6]:
all_convos = [item for split in raw.values() for item in split]
by_id = {int(item["convo_id"]): item for item in all_convos}


def action_names(item: dict) -> tuple[str, ...]:
    names = []
    for turn in item.get("delexed") or []:
        if turn.get("speaker") == "action":
            targets = turn.get("targets") or []
            if len(targets) > 2 and targets[2]:
                names.append(str(targets[2]))
    return tuple(names)


def first_customer(item: dict) -> str:
    for speaker, text in item["original"]:
        if speaker == "customer":
            return text.strip()
    return ""


for flow in SHORTLIST:
    print(f"\n==== {flow} top observed action sequences ====")
    grouped: dict[str, list] = defaultdict(list)
    for item in all_convos:
        if item["scenario"]["flow"] == flow:
            grouped[item["scenario"]["subflow"]].append(item)
    for subflow, items in grouped.items():
        counts = Counter(action_names(c) for c in items)
        top_seq, top_n = counts.most_common(1)[0]
        print(f"  {subflow:20s} unique_seqs={len(counts):3d}  mode_n={top_n:3d}  {top_seq}")


==== account_access top observed action sequences ====
  recover_username     unique_seqs= 31  mode_n=297  ('pull-up-account', 'verify-identity')
  recover_password     unique_seqs= 31  mode_n=183  ('pull-up-account', 'enter-details', 'make-password')
  reset_2fa            unique_seqs= 36  mode_n=272  ('pull-up-account', 'enter-details', 'send-link')

==== shipping_issue top observed action sequences ====
  manage               unique_seqs= 81  mode_n= 89  ('pull-up-account', 'shipping-status', 'validate-purchase', 'update-order')
  status               unique_seqs= 97  mode_n= 73  ('pull-up-account', 'verify-identity', 'validate-purchase', 'ask-the-oracle')
  missing              unique_seqs= 85  mode_n= 57  ('pull-up-account', 'validate-purchase', 'record-reason', 'update-order', 'make-purchase')
  cost                 unique_seqs= 82  mode_n= 74  ('pull-up-account', 'validate-purchase', 'shipping-status', 'offer-refund')

==== product_defect top observed action sequences ====
  re

## 4. Excerpts (why language matters)

BERTopic will see original customer/agent text, **not** action-button names (those would leak subflow identity).

In [7]:
EXCERPT_IDS = {
    "account_access / recover_password": 2652,
    "account_access / reset_2fa": 7225,
    "shipping_issue / missing": 169,
    "shipping_issue / cost": 194,
    "shipping_issue / manage": 264,
    "shipping_issue / status": 228,
    "product_defect / return_stain": None,
}


def show_excerpt(item: dict, n_turns: int = 6) -> None:
    print(f"id={item['convo_id']} {item['scenario']['flow']}/{item['scenario']['subflow']}")
    print("actions", list(action_names(item)))
    for speaker, text in item["original"][:n_turns]:
        print(f"  {speaker:10s} {text[:140]}")
    print()


for item in all_convos:
    if item["scenario"]["flow"] == "product_defect" and item["scenario"]["subflow"] == "return_stain":
        EXCERPT_IDS["product_defect / return_stain"] = int(item["convo_id"])
        break

for label, convo_id in EXCERPT_IDS.items():
    print(f"-- {label} --")
    show_excerpt(by_id[convo_id])

-- account_access / recover_password --
id=2652 account_access/recover_password
actions ['pull-up-account', 'verify-identity', 'enter-details', 'make-password']
  agent      Thank you for contacting AcmeBrands! How can I help you today?
  customer   Hi!  I'm trying to log in but can't remember my password.
  agent      I'm sorry you having problems signing in to the website, but I'll be happy to help. Could I have your name and account ID please?
  customer   My name is Chloe Zhang.  I don't have my account ID but I can give you my address or phone number
  agent      That's okay. I'm sure I can locate your account with you name. One moment please.
  action     Account has been pulled up for Chloe Zhang.

-- account_access / reset_2fa --
id=7225 account_access/reset_2fa
actions ['pull-up-account', 'enter-details', 'send-link']
  agent      Hello, how may I help you today?
  customer   Hi! I was trying to log into my account but I have lost the phone that I use for two-factor authentica

## 5. Rejected subsets

**`storewide_query` / `single_item_query`:** FAQ search-button workflows. Ontology lists 4 storewide subflows; the data actually stores `pricing_1`…`policy_4`. Language is catalog Q&A, not an operational playbook the agent would maintain week to week.

**`account_access`:** Best language separation (username vs password vs 2FA) and short, readable guidelines. **Rejected as the demo flow** because it has only three subflows, so Week 2 cannot introduce a fourth held-out workflow from the same top-level flow.

**`product_defect`:** Six subflows, but `return_stain` / `return_color` / `return_size` share the same canonical actions and nearly identical guideline text. Picking four would still leave two return variants that are not operationally distinct.

**`shipping_issue`:** Exactly four subflows, four different procedures (change shipment, shipping-fee complaint, never-arrived reship, email/status discrepancy), and enough conversations each (~240+). Language overlaps more than account_access — that is the honest BERTopic test.

In [8]:
display(
    con.execute(
        """
        SELECT flow, count(DISTINCT subflow) AS n_subflows, count(*) AS n
        FROM abcd_index
        WHERE flow IN ('storewide_query', 'single_item_query', 'account_access',
                       'product_defect', 'shipping_issue')
        GROUP BY 1
        ORDER BY 1
        """
    ).df()
)
print("storewide_query actual subflows", 
      con.execute("SELECT DISTINCT subflow FROM abcd_index WHERE flow='storewide_query' ORDER BY 1").fetchall()[:8], "...")

,flow,n_subflows,n
0,account_access,3,1048
1,product_defect,6,1070
2,shipping_issue,4,1020
3,single_item_query,32,1045
4,storewide_query,16,1094


storewide_query actual subflows [('membership_1',), ('membership_2',), ('membership_3',), ('membership_4',), ('policy_1',), ('policy_2',), ('policy_3',), ('policy_4',)] ...


## 6. BERTopic feasibility

Settings kept small and documented (not a topic-model research pass):

- documents = original customer+agent text (no action names)
- embedding = `all-MiniLM-L6-v2`
- UMAP `random_state=42`, `min_dist=0.0`, `metric=cosine`
- HDBSCAN `min_samples=1`, `cluster_selection_method=eom`
- CountVectorizer with extra stopwords (`agent`, `customer`, `help`, `account`, `order`, `id`, …) so descriptors are not dominated by chat boilerplate

We do **not** require `topic_id == subflow`. Success = a human can read a topic descriptor + representatives and recognize a recurring operational pattern.

In [9]:
def balanced_sample(flow: str, per_subflow: int) -> list[dict]:
    grouped: dict[str, list] = defaultdict(list)
    for item in all_convos:
        if item["scenario"]["flow"] == flow:
            grouped[item["scenario"]["subflow"]].append(item)
    picked = []
    for subflow, items in sorted(grouped.items()):
        items = sorted(items, key=lambda c: int(c["convo_id"]))
        picked.extend(items[:per_subflow])
    return picked


def topic_crosstab(items: list[dict], result) -> None:
    labels = [item["scenario"]["subflow"] for item in items]
    print("topic sizes", Counter(result.assignments).most_common())
    print("config", {k: result.config.model_dump()[k] for k in ("min_cluster_size", "n_neighbors", "seed", "embedding_model")})
    by_topic: dict[int, Counter] = defaultdict(Counter)
    for topic_id, label in zip(result.assignments, labels):
        by_topic[topic_id][label] += 1
    descriptors = {topic.topic_id: topic.descriptor for topic in result.topics}
    for topic_id in sorted(by_topic):
        words = [part.strip() for part in descriptors.get(topic_id, "").split(",") if part.strip()][:6]
        print(f"  topic {topic_id:3d} n={sum(by_topic[topic_id].values()):3d}  {words}  {by_topic[topic_id].most_common()}")


print("extra stopwords", sorted(EXTRA_STOP))

extra stopwords ['account', 'acme', 'acmebrands', 'agent', 'customer', 'day', 'email', 'full', 'good', 'great', 'hello', 'help', 'hi', 'id', 'let', 'moment', 'name', 'need', 'no', 'ok', 'okay', 'one', 'order', 'please', 'thank', 'thanks', 'today', 'username', 'want', 'welcome', 'yes']


### 6a. `account_access` — cleaner clusters, cannot supply a fourth subflow

In [10]:
access = balanced_sample("account_access", 40)
access_result = discover_topics(
    [parse_raw_conversation(item) for item in access],
    config=BertopicConfig(min_cluster_size=8, n_neighbors=12, min_to_cluster=2),
)
print("account_access n=120")
topic_crosstab(access, access_result)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

account_access n=120
topic sizes [(0, 94), (1, 26)]
config {'min_cluster_size': 8, 'n_neighbors': 12, 'seed': 42, 'embedding_model': 'all-MiniLM-L6-v2'}
  topic   0 n= 94  ['password', 'number', 'new', 'phone', 'address', 'pin']  [('recover_password', 40), ('recover_username', 40), ('reset_2fa', 14)]
  topic   1 n= 26  ['reset', 'phone', 'factor', 'authentication', 'send', 'factor authentication']  [('reset_2fa', 26)]


Password / username / 2FA fall into mostly-pure topics. Usable, but Week 2 has no in-flow `D`.

### 6b. `product_defect` — returns collapse

In [11]:
defect = balanced_sample("product_defect", 30)
defect_result = discover_topics(
    [parse_raw_conversation(item) for item in defect],
    config=BertopicConfig(min_cluster_size=8, n_neighbors=12, min_to_cluster=2),
)
print("product_defect n=180")
topic_crosstab(defect, defect_result)

product_defect n=180
topic sizes [(0, 99), (1, 81)]
config {'min_cluster_size': 8, 'n_neighbors': 12, 'seed': 42, 'embedding_model': 'all-MiniLM-L6-v2'}
  topic   0 n= 99  ['return', 'address', 'like', 'purchase', 'mail', 'membership']  [('return_color', 30), ('return_size', 29), ('return_stain', 29), ('refund_initiate', 7), ('refund_update', 3), ('refund_status', 1)]
  topic   1 n= 81  ['refund', 'card', 'like', 'check', 'status', 'credit']  [('refund_status', 29), ('refund_update', 27), ('refund_initiate', 23), ('return_size', 1), ('return_stain', 1)]


The three return reasons merge. Refund initiate/update also mix. Not four demo-distinct workflows.

### 6c. `shipping_issue` — merges and splits, but descriptors are operational

In [12]:
shipping = balanced_sample("shipping_issue", 40)
shipping_result = discover_topics(
    [parse_raw_conversation(item) for item in shipping],
    config=BertopicConfig(min_cluster_size=8, n_neighbors=12, min_to_cluster=2),
)
print("shipping_issue n=160")
topic_crosstab(shipping, shipping_result)

shipping_issue n=160
topic sizes [(0, 60), (1, 30), (2, 26), (3, 18), (4, 11), (5, 8), (-1, 7)]
config {'min_cluster_size': 8, 'n_neighbors': 12, 'seed': 42, 'embedding_model': 'all-MiniLM-L6-v2'}
  topic  -1 n=  7  []  [('manage', 5), ('missing', 1), ('status', 1)]
  topic   0 n= 60  ['shipping', 'refund', 'cancel', 'status', 'like', 'cost']  [('cost', 38), ('manage', 14), ('status', 7), ('missing', 1)]
  topic   1 n= 30  ['ordered', 'sorry', 'shirt', 'item', 'address', 'product']  [('missing', 17), ('status', 11), ('manage', 2)]
  topic   2 n= 26  ['date', 'status', 'shipping', 'check', 'days', 'tomorrow']  [('missing', 13), ('status', 11), ('cost', 1), ('manage', 1)]
  topic   3 n= 18  ['address', 'change', 'new address', 'new', 'update', 'old']  [('manage', 18)]
  topic   4 n= 11  ['days', 'package', 'sure', 'just', 'shipping', 'status']  [('missing', 8), ('status', 2), ('cost', 1)]
  topic   5 n=  8  ['address', 'shipping address', 'correct', 'confirm', 'correct address', 'shippin

Typical pattern (exact topic IDs will jitter slightly with UMAP): a **cost/refund/cancel** topic, a **missing/waiting** topic, a **change-address** topic, and status split between “check shipment” and “confirm address”. That is good enough for a coverage-judgment demo. We will not chase purity.

### 6d. Week-sized batch (the actual runtime scale)

A 13-conversation batch with `min_cluster_size=2` **fragmented**. ~18–20 conversations with `min_cluster_size=4` is the smallest setting that still produced recurring topics rather than pairs.

In [13]:
week1_raw = []
for role, ids in WEEK_MEMBERSHIP["week_1"].items():
    week1_raw.extend(by_id[i] for i in ids)
week1_result = discover_topics(
    [parse_raw_conversation(item) for item in week1_raw],
    config=BertopicConfig(min_cluster_size=4, n_neighbors=6, min_to_cluster=2, min_topic_n=1),
)
print("pinned week_1 n=", len(week1_raw))
topic_crosstab(week1_raw, week1_result)
print("\nrepresentatives / descriptors")
for topic in week1_result.topics:
    if topic.topic_id == -1:
        continue
    print(f"\ntopic {topic.topic_id} {topic.descriptor}")
    for conversation_id in topic.representative_ids:
        item = by_id[int(conversation_id)]
        print(f"  [{item['scenario']['subflow']}] {first_customer(item)[:120]}")

pinned week_1 n= 18
topic sizes [(0, 10), (1, 8)]
config {'min_cluster_size': 4, 'n_neighbors': 6, 'seed': 42, 'embedding_model': 'all-MiniLM-L6-v2'}
  topic   0 n= 10  ['shipping', 'status', 'cancel', 'like', 'address', 'know']  [('manage', 5), ('cost', 5)]
  topic   1 n=  8  ['address', 'days', 'package', 'jacket', 'sorry', 'product']  [('missing', 8)]

representatives / descriptors

topic 0 shipping, status, cancel, like, address, know, sure, refund
  [manage] Hello, I need to update the shipping address on my order
  [manage] Hi I need to change my shipping details
  [manage] Hi, I ordered something but I need to change my shipping details.

topic 1 address, days, package, jacket, sorry, product, com, new
  [missing] Hi. I never received my package. Can I please get it resent?
  [missing] I ordered this jacket over a month ago and I still haven't received it.
  [missing] Hey! I have a missing package? It was supposed to be delivered 5 days ago.


Week 1 is expected to surface at least a **missing-package** topic (held-out `C`) and a **shipping-cost** topic (seed `B`). `manage` (`A`) may split into the others — acceptable noise. Downstream coverage judgment uses retrieval, not topic-id equality.

## 7. Chosen experiment

| role | subflow | week 1 | week 2 | week 3 | seed KB (later) |
| --- | --- | --- | --- | --- | --- |
| A | `manage` | yes | yes | yes | covered |
| B | `cost` | yes | yes | yes | covered |
| C | `missing` | recurring | still present | present | **held out** |
| D | `status` | absent | **introduced** | present | **held out** |

Flow: **`shipping_issue`**. BERTopic settings above are the Phase 1 default (HDBSCAN, not KMeans). Custom clustering is **not** justified — HDBSCAN already yields readable recurring candidates on this subset.

In [14]:
print("flow", DEMO_FLOW)
print("roles", ROLE_TO_SUBFLOW)
for week_id, roles in WEEK_MEMBERSHIP.items():
    counts = {role: len(ids) for role, ids in roles.items() if ids}
    print(week_id, counts, "n=", sum(counts.values()))

flow shipping_issue
roles {'A': 'manage', 'B': 'cost', 'C': 'missing', 'D': 'status'}
week_1 {'A': 5, 'B': 5, 'C': 8} n= 18
week_2 {'A': 3, 'B': 3, 'C': 3, 'D': 8} n= 17
week_3 {'A': 4, 'B': 4, 'C': 4, 'D': 4} n= 16


## 8. Hidden-label week membership (offline)

These labels must not appear on `load_week()`.

In [15]:
if not (ROOT / "data" / "demo" / "weeks.parquet").exists():
    subprocess.check_call([sys.executable, str(ROOT / "scripts" / "prepare_demo.py")])

labels = load_fixture_labels()
rows = [
    {
        "week_id": label.week_id,
        "role": label.role,
        "subflow": label.subflow,
        "conversation_id": label.conversation_id,
    }
    for label in labels
]
con.register("fixture_labels", pa.Table.from_pylist(rows))
display(
    con.execute(
        """
        SELECT week_id, role, subflow, count(*) AS n
        FROM fixture_labels
        GROUP BY 1, 2, 3
        ORDER BY week_id, role
        """
    ).df()
)
display(
    con.execute(
        """
        SELECT week_id, list(conversation_id ORDER BY conversation_id) AS conversation_ids
        FROM fixture_labels
        GROUP BY 1
        ORDER BY 1
        """
    ).df()
)

,week_id,role,subflow,n
0,week_1,A,manage,5
1,week_1,B,cost,5
2,week_1,C,missing,8
3,week_2,A,manage,3
4,week_2,B,cost,3
5,week_2,C,missing,3
6,week_2,D,status,8
7,week_3,A,manage,4
8,week_3,B,cost,4
9,week_3,C,missing,4


,week_id,conversation_ids
0,week_1,"[1226, 169, 1821, 1877, 194, 1944, 264, 450, 4..."
1,week_2,"[1012, 1043, 1128, 1439, 1558, 1582, 1914, 203..."
2,week_3,"[1031, 1162, 1175, 1419, 1466, 1503, 1722, 186..."


## 9. Runtime contract: unlabeled weekly batches

In [16]:
for week_id in ("week_1", "week_2", "week_3"):
    batch = load_week(week_id)
    sample = batch.conversations[0]
    print(
        week_id,
        "n=",
        len(batch.conversation_ids),
        "fields=",
        set(sample.model_dump()),
        "n_actions=",
        len(sample.actions),
        "doc_chars=",
        len(conversation_document(sample)),
    )
    assert "subflow" not in sample.model_dump()
    assert "flow" not in sample.model_dump()

print("first customer turn, week_1 sample:")
print(next(t.text for t in load_week("week_1").conversations[0].turns if t.speaker == "customer"))

week_1 n= 18 fields= {'conversation_id', 'actions', 'turns'} n_actions= 3 doc_chars= 683
week_2 n= 17 fields= {'conversation_id', 'actions', 'turns'} n_actions= 4 doc_chars= 832
week_3 n= 16 fields= {'conversation_id', 'actions', 'turns'} n_actions= 4 doc_chars= 768
first customer turn, week_1 sample:
hi, i need to change my shipping details so i can receive my items at the new address


## 10. Tests + LangSmith env (no eval dataset yet)

In [17]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_demo_fixtures.py"],
    cwd=ROOT,
    check=False,
    capture_output=True,
    text=True,
)
print(result.stdout)
print(result.stderr)
print("pytest exit", result.returncode)
assert result.returncode == 0

print("LANGSMITH_TRACING", os.getenv("LANGSMITH_TRACING"))
print("LANGSMITH_PROJECT", os.getenv("LANGSMITH_PROJECT"))
print("LANGSMITH_API_KEY set", bool(os.getenv("LANGSMITH_API_KEY")))
print("OPENAI_API_KEY set", bool(os.getenv("OPENAI_API_KEY")))
try:
    from langsmith import Client

    if os.getenv("LANGSMITH_API_KEY"):
        Client()
        print("LangSmith Client() constructed")
    else:
        print("LangSmith key absent — skip Client ping")
except ImportError:
    print("langsmith not installed in Phase 1 — env placeholders only")

.......                                                                  [100%]
7 passed in 1.29s


pytest exit 0
LANGSMITH_TRACING true
LANGSMITH_PROJECT fresh-skills
LANGSMITH_API_KEY set True
OPENAI_API_KEY set True
langsmith not installed in Phase 1 — env placeholders only


## 11. Phase 1 decision (for merge review)

- **Flow:** `shipping_issue`
- **A/B/C/D:** `manage` / `cost` / `missing` / `status`
- **BERTopic:** off-the-shelf, usable on ~20-conversation weeks with the stopword + `min_cluster_size=4` settings above. Expect merges (status↔missing) and splits (manage). Do not unit-test topic IDs.
- **Refactor:** `NO REFACTOR`
- **Not done here:** seed KB, RAG, graph, HITL